In [ ]:
!pip install -U langchain langchain-community langchain-core


In [ ]:
# pip install langchain langchain-community
from langchain_core.prompts import PromptTemplate

# Before: hardcoded f-string (bad practice)
# def explain_topic(topic): return f"Explain {topic} in simple terms for beginners"

# After: reusable LangChain PromptTemplate
explain_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners."
)

# Use .format() to inject values — no f-strings needed
prompt = explain_template.format(topic="Machine Learning")
print(prompt)
# → Explain Machine Learning in simple terms for beginners.

Explain Machine Learning in simple terms for beginners.


In [ ]:
from langchain_core.prompts import PromptTemplate

# Template with 3 dynamic inputs
multi_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone."
)

# Test cases
test_cases = [
    {"topic": "AI",            "audience": "beginners", "tone": "friendly"},
    {"topic": "Python",        "audience": "kids",      "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers", "tone": "technical"},
]

for tc in test_cases:
    print(multi_template.format(**tc))

Explain AI for beginners in a friendly tone.
Explain Python for kids in a fun tone.
Explain Deep Learning for engineers in a technical tone.


In [ ]:
from langchain_core.prompts import PromptTemplate

# Three distinct prompt templates for the same topic
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step."
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 interview questions about {topic}."
)

storytelling_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as an engaging story."
)

topic = "Machine Learning"

print("--- Teaching ---")
print(teaching_template.format(topic=topic))

print("\n--- Interview ---")
print(interview_template.format(topic=topic))

print("\n--- Storytelling ---")
print(storytelling_template.format(topic=topic))

--- Teaching ---
Explain Machine Learning clearly step by step.

--- Interview ---
Ask 3 interview questions about Machine Learning.

--- Storytelling ---
Explain Machine Learning as an engaging story.


In [ ]:
# Add ChatPromptTemplate to your imports
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

role_instructions = {
    "teacher":     "You are a patient and clear teacher who explains concepts step by step.",
    "interviewer": "You are a strict interviewer asking technical questions.",
    "motivator":   "You are an enthusiastic motivator who makes learning exciting.",
}

def build_chat_prompt(role: str, topic: str):
    # This now works because ChatPromptTemplate is imported
    system_msg = SystemMessagePromptTemplate.from_template(role_instructions[role])
    human_msg  = HumanMessagePromptTemplate.from_template("Tell me about {topic}.")

    chat_prompt = ChatPromptTemplate.from_messages([system_msg, human_msg])

    messages = chat_prompt.format_messages(topic=topic)
    for msg in messages:
        print(f"[{msg.__class__.__name__}]: {msg.content}")

# Example
build_chat_prompt(role="teacher", topic="Neural Networks")


[SystemMessage]: You are a patient and clear teacher who explains concepts step by step.
[HumanMessage]: Tell me about Neural Networks.


In [ ]:
VALID_AUDIENCES = ["beginner", "intermediate", "expert"]
VALID_TONES     = ["formal", "casual", "fun"]

def validate_inputs(audience: str, tone: str) -> tuple:
    """
    Validates audience and tone inputs.
    Returns corrected values or raises ValueError on bad input.
    """
    audience = audience.lower().strip()
    tone     = tone.lower().strip()

    # Assign defaults if invalid (soft validation)
    if audience not in VALID_AUDIENCES:
        print(f"Warning: '{audience}' is invalid. Defaulting to 'beginner'.")
        audience = "beginner"

    if tone not in VALID_TONES:
        print(f"Warning: '{tone}' is invalid. Defaulting to 'casual'.")
        tone = "casual"

    return audience, tone

# Test
print(validate_inputs("kids", "friendly"))   # → defaults applied
print(validate_inputs("expert", "formal"))   # → valid, passes through

('beginner', 'casual')
('expert', 'formal')


In [ ]:
from langchain_core.prompts import PromptTemplate

# Style-to-template mapping
STYLE_TEMPLATES = {
    "teaching":     "Explain {topic} for {audience} in a {tone} tone, step by step.",
    "interview":    "You are interviewing a {audience}. Ask 3 questions about {topic} in a {tone} tone.",
    "storytelling": "Tell {topic} as a story for {audience} in a {tone} tone.",
}

def generate_prompt(topic: str, audience: str, tone: str, style: str) -> str:
    """
    Generates a dynamic prompt using a PromptTemplate.

    Args:
        topic    : Subject to explain (e.g., "Neural Networks")
        audience : Target learner (e.g., "beginners")
        tone     : Communication style (e.g., "fun")
        style    : Prompt variant — teaching | interview | storytelling

    Returns:
        Formatted prompt string ready to send to an LLM.
    """
    # Validate first
    audience, tone = validate_inputs(audience, tone)

    if style not in STYLE_TEMPLATES:
        raise ValueError(f"Style must be one of: {list(STYLE_TEMPLATES.keys())}")

    template = PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template=STYLE_TEMPLATES[style]
    )

    return template.format(topic=topic, audience=audience, tone=tone)


# Example output
result = generate_prompt(
    topic="Neural Networks",
    audience="beginners",
    tone="fun",
    style="storytelling"
)
print(result)
# → Tell Neural Networks as a story for beginners in a fun tone.

Tell Neural Networks as a story for beginner in a fun tone.


In [ ]:
from langchain_core.prompts import PromptTemplate

# Single template used across 5 completely different inputs
universal_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} to {audience} using a {tone} tone."
)

# 5 varied inputs — same structure, different outputs
test_inputs = [
    {"topic": "Photosynthesis",    "audience": "10-year-olds",  "tone": "fun"},
    {"topic": "Blockchain",        "audience": "investors",     "tone": "formal"},
    {"topic": "Recursion",         "audience": "CS students",   "tone": "technical"},
    {"topic": "Climate Change",    "audience": "policymakers",  "tone": "serious"},
    {"topic": "Generative AI",     "audience": "HR managers",   "tone": "casual"},
]

print("=== Reusability Test: One Template, Five Outputs ===\n")
for i, inputs in enumerate(test_inputs, 1):
    print(f"Input {i}: {inputs}")
    print(f"Output:  {universal_template.format(**inputs)}\n")

=== Reusability Test: One Template, Five Outputs ===

Input 1: {'topic': 'Photosynthesis', 'audience': '10-year-olds', 'tone': 'fun'}
Output:  Explain Photosynthesis to 10-year-olds using a fun tone.

Input 2: {'topic': 'Blockchain', 'audience': 'investors', 'tone': 'formal'}
Output:  Explain Blockchain to investors using a formal tone.

Input 3: {'topic': 'Recursion', 'audience': 'CS students', 'tone': 'technical'}
Output:  Explain Recursion to CS students using a technical tone.

Input 4: {'topic': 'Climate Change', 'audience': 'policymakers', 'tone': 'serious'}
Output:  Explain Climate Change to policymakers using a serious tone.

Input 5: {'topic': 'Generative AI', 'audience': 'HR managers', 'tone': 'casual'}
Output:  Explain Generative AI to HR managers using a casual tone.



In [ ]:
# Full pipeline demo — Tasks 5 + 6 together
print("=== Mini Prompt Engine Demo ===\n")

demo_cases = [
    ("Deep Learning", "beginners", "fun",       "storytelling"),
    ("Python",        "kids",      "casual",     "teaching"),
    ("Transformers",  "engineers", "technical",  "interview"),
    ("AI Ethics",     "managers",  "formal",     "teaching"),
]

for topic, audience, tone, style in demo_cases:
    prompt = generate_prompt(topic, audience, tone, style)
    print(f"Topic={topic!r}, Audience={audience!r}, Tone={tone!r}, Style={style!r}")
    print(f"  → {prompt}\n")

=== Mini Prompt Engine Demo ===

Topic='Deep Learning', Audience='beginners', Tone='fun', Style='storytelling'
  → Tell Deep Learning as a story for beginner in a fun tone.

Topic='Python', Audience='kids', Tone='casual', Style='teaching'
  → Explain Python for beginner in a casual tone, step by step.

Topic='Transformers', Audience='engineers', Tone='technical', Style='interview'
  → You are interviewing a beginner. Ask 3 questions about Transformers in a casual tone.

Topic='AI Ethics', Audience='managers', Tone='formal', Style='teaching'
  → Explain AI Ethics for beginner in a formal tone, step by step.

